In [2]:

import cv2
import matplotlib.pyplot as plt
import json
import numpy as np
import yaml
import os
from tqdm import tqdm
from scipy.spatial.distance import euclidean
import pandas as pd
import copy
from sklearn.preprocessing import StandardScaler
import math


In [32]:
config_path = os.path.join('config.yaml')

with open(config_path) as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)
    
path_keypoints_3d_1person = cfg['path_keypoints_3d_1person']
connections = cfg['connections']
useless_points = cfg['coco_head_points']
base_connestion = cfg['base_connestion']
angles = cfg['angles']

In [33]:

data_list = []

# Iteracja przez pliki w folderze
for filename in tqdm(os.listdir(path_keypoints_3d_1person), desc="Wczytywanie plików .info"):
    if filename.endswith('.info'):
        file_path = os.path.join(path_keypoints_3d_1person, filename)

        # Wczytanie pliku JSON i dodanie do listy
        with open(file_path, 'r', encoding='utf-8') as file:
            json_data = json.load(file)[0]
            json_data['filename'] = filename
        
        # Dodanie słownika do listy
        data_list.append(json_data)

# Utworzenie DataFrame z listy słowników
keypoints_df = pd.DataFrame(data_list)

# czy to pomaga w notebookach?
del data_list

Wczytywanie plików .info: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 55746/55746 [12:45<00:00, 72.85it/s]


In [44]:
def remove_Points(df, points):
    df['keypoints_clear'] = df['keypoints'].apply(lambda x: [point for i, point in enumerate(x) if i not in points])
    df['keypoint_scores_clear'] = df['keypoint_scores'].apply(lambda x: [score for i, score in enumerate(x) if i not in points])
    return df

In [45]:
#keypoints_df_clear = delete_Points(keypoints_df, useless_points)

In [46]:

def count_lenghts(keypoints, connections):
    lenghts = [euclidean(keypoints[point1], keypoints[point2]) for point1, point2 in connections]
    return lenghts

def add_lenghts(df, connections, useless_points=[]):
    connections = [(x, y) for x, y in connections if x not in useless_points and y not in useless_points]
    df['lengths'] = df['keypoints'].apply(lambda x: count_lenghts(x, connections))
    

In [47]:
add_lenghts(keypoints_df, connections, useless_points=useless_points)

In [48]:
def add_relative_lenghts(df, base_connestion):
    df['base_length'] = df['keypoints'].apply(lambda x: count_lenghts(x, [base_connestion])[0])
    # gdyby oba stawy w tym samym pixelu (w 1percent 236 razy)
    df['base_length'] = df['base_length'].apply(lambda x: 1 if x < 1 else x)
    df['relative_lengths'] = df.apply(lambda row: [length / row['base_length'] for length in row['lengths']], axis=1)

In [49]:
add_relative_lenghts(keypoints_df, base_connestion)

In [51]:
import pandas as pd

def change_value(list_to_change, index, value):
    list_to_change[index]= value

    return list_to_change

def std_index(kolumna, index): 
    elements = kolumna.apply(lambda x: x[index])
    elements_mean = elements.mean()
    elements_std = elements.std()
    elements = [(elem - elements_mean) / elements_std for elem in elements]
    return kolumna.apply(lambda x: change_value(x, index, elements[index]))
 
def std_lengths(df):
# Standaryzacja dla wszystkich indeksów
    lenghts_list_len = len(df['relative_lengths'][0])
    # Załóż, że wszystkie listy mają tę samą długość
    df['relative_lengths_std'] = df['relative_lengths'].apply(copy.deepcopy)
    for i in range(lenghts_list_len):
        
        df['relative_lengths_std'] = std_index(df['relative_lengths_std'], i)

# Wyświetlenie zmodyfikowanej ramki danych
std_lengths(keypoints_df)

In [87]:
def angle3pt(a, b, c):
    # Oblicz różnice współrzędnych dla każdej osi
    delta_x1, delta_y1, delta_z1 = a[0] - b[0], a[1] - b[1], a[2] - b[2]
    delta_x2, delta_y2, delta_z2 = c[0] - b[0], c[1] - b[1], c[2] - b[2]

    # Oblicz kąty azymutu (phi) i elewacji (theta) dla obu wektorów
    theta1 = math.atan2(delta_y1, math.sqrt(delta_x1**2 + delta_z1**2))
    phi1 = math.atan2(delta_z1, delta_x1)
    
    theta2 = math.atan2(delta_y2, math.sqrt(delta_x2**2 + delta_z2**2))
    phi2 = math.atan2(delta_z2, delta_x2)

    # Oblicz kąt między wektorami (różnice kątów azymutu i elewacji)
    angle_rad = math.acos(math.cos(theta1 - theta2) * math.cos(phi1 - phi2))

    # Konwertuj kąt z radianów na stopnie
    ang = math.degrees(angle_rad)
    return ang + 360 if ang < 0 else ang

def get_base_angle(mid_point, end_point, three_dim=False):
    base_end_point = [0.0, mid_point[1], mid_point[2]] if three_dim else [0.0, mid_point[1]]
    return angle3pt(base_end_point, mid_point, end_point)

# dla base_points nie działa tak jak powinno
def get_angle3(a, b, c, base_points=None, three_dim=False):
    if base_points:
        base_angle = get_base_angle(base_points[0], base_points[1], three_dim)
    else:
        base_angle = 0.0
    angle = angle3pt(a, b, c) - base_angle
    return angle + 360 if angle < 0 else angle

In [88]:
def count_angles(keypoints, angles, base_points=None, three_dim=False):
    angles_size = [get_angle3(keypoints[point1], keypoints[point2], keypoints[point3], base_points, three_dim) for point1, point2, point3 in angles]
    return angles_size

def add_angles(df, angles, useless_points=[], base_connestion=None, three_dim=False):
    angles = [(x, y, z) for x, y, z in angles if x not in useless_points and y not in useless_points]
    df['angles'] = df['keypoints'].apply(lambda x: count_angles(x, angles, base_connestion, three_dim))

In [89]:
add_angles(keypoints_df, angles, three_dim=True)

In [83]:
def count_variant_angle(keypoints, connections, three_dim=False):
    lenghts = [get_base_angle(keypoints[point1], keypoints[point2], three_dim=three_dim) for point1, point2 in connections]
    return lenghts

def add_variant_angle(df, connections, useless_points=[], three_dim=False):
    connections = [(x, y) for x, y in connections if x not in useless_points and y not in useless_points]
    df['variant_angles'] = df['keypoints'].apply(lambda x: count_variant_angle(x, connections, three_dim))

In [90]:
add_variant_angle(keypoints_df, connections, three_dim=True)

In [91]:
keypoints_df.head()

,keypoints,keypoint_scores,filename,lengths,base_length,relative_lengths,relative_lengths_std,angles,variant_angles
0,"[[-0.0, 0.0, 0.2702668309211731], [-0.10712616...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",21andhungry-1709004344122507373-pose_3d.info,"[0.33018780308021894, 0.057105861195240415, 0....",1.0,"[0.33018780308021894, 0.057105861195240415, 0....","[2.296354321189886, -0.3466812650788048, 2.927...","[41.39361674875644, 3.9853043646494832, 6.1690...","[169.38938492088474, 57.16544063467, 166.25267..."
1,"[[-0.0, 0.0, 0.18142542243003845], [-0.0297661...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",21andhungry-1732757448640221464-pose_3d.info,"[0.015933208563062552, 0.22181455680245732, 0....",1.0,"[0.015933208563062552, 0.22181455680245732, 0....","[2.296354321189886, -0.3466812650788048, 2.927...","[13.649235319983353, 12.88462546533579, 13.538...","[106.49173983389692, 61.4002672881022, 88.1452..."
2,"[[-0.0, 0.0, 0.21224242448806763], [0.01438415...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",21andhungry-1737839725787390113-pose_3d.info,"[0.01430244203653737, 0.47553243277448587, 0.3...",1.0,"[0.01430244203653737, 0.47553243277448587, 0.3...","[2.296354321189886, -0.3466812650788048, 2.927...","[5.098250279950368, 7.3700776943762145, 2.3063...","[85.33118323487774, 90.18227322436202, 87.1544..."
3,"[[-0.0, 0.0, 0.2062559872865677], [-0.12660333...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",21andhungry-1743867257748275198-pose_3d.info,"[0.2921167645762577, 0.030920752326902487, 0.2...",1.0,"[0.2921167645762577, 0.030920752326902487, 0.2...","[2.296354321189886, -0.3466812650788048, 2.927...","[2.2118199120721203, 2.760563402826584, 1.7961...","[176.21682879043038, 126.9783931796221, 166.44..."
4,"[[-0.0, 0.0, 0.1800045222043991], [0.045351866...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",21andhungry-1748873849862474760-pose_3d.info,"[0.015743644352326287, 0.26351504744439747, 0....",1.0,"[0.015743644352326287, 0.26351504744439747, 0....","[2.296354321189886, -0.3466812650788048, 2.927...","[148.74223009849962, 162.72143783375583, 108.7...","[70.41269653489648, 53.27225986232498, 70.4324..."
